# Webull Thailand Endpoint Tutorial: Order Preview Guardrails

    Learning goals:
    - รู้ว่า endpoint ในหมวดนี้ใช้ทำอะไร
    - เริ่มจาก offline sample response ก่อนใช้ credential จริง
    - บันทึก raw JSON ทุก endpoint เพื่อกลับมาตรวจ response ได้
    - แปลง response เป็น DataFrame หรือกราฟเมื่อเหมาะสม

    Official sources:
    - https://developer.webull.co.th/apis/docs/reference/trade-api/trading/
- https://developer.webull.co.th/apis/docs/reference/trade-api/common-order-preview/

    Endpoints covered:
    - `POST /openapi/trade/order/preview` - Order Preview


## Safety Defaults

ค่า default คือ offline mode (`WEBULL_TUTORIAL_LIVE=0`) จึง run ได้โดยไม่ต้องมี credential.
ถ้าจะยิง API จริง ให้ตั้งค่า `.env.webull-th` หรือ `.env` แล้วเปิด `WEBULL_TUTORIAL_LIVE=1`.


In [ ]:
NOTEBOOK_SLUG = "05-order-preview-guardrails"
ENDPOINTS = [{'name': 'order_preview',
  'label': 'Order Preview',
  'method': 'POST',
  'path': '/openapi/trade/order/preview',
  'purpose': 'Estimate cost and amount for a simple order.',
  'params': {},
  'body': {'account_id': 'from_env_or_sample',
           'symbol': 'AAPL',
           'category': 'US_STOCK',
           'side': 'BUY',
           'order_type': 'LIMIT',
           'quantity': '1',
           'limit_price': '250.00',
           'time_in_force': 'DAY'}}]


In [ ]:
import base64
import hashlib
import hmac
import json
import os
import uuid
from datetime import UTC, datetime
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests

try:
    NOTEBOOK_DISPLAY = display
except NameError:
    def display(value: object) -> None:
        print(value)
else:
    display = NOTEBOOK_DISPLAY

BASE_URL = "https://api.webull.co.th"
HOST = "api.webull.co.th"
REGION = "th"
LIVE_MODE = os.getenv("WEBULL_TUTORIAL_LIVE", "0") == "1"
OUTPUT_ROOT = Path(os.getenv("WEBULL_TUTORIAL_OUTPUT_DIR", "outputs/webull-th-endpoints"))
OUTPUT_DIR = OUTPUT_ROOT / NOTEBOOK_SLUG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_env_file(path: str | Path) -> dict[str, str]:
    env_path = Path(path)
    if not env_path.exists():
        return {}
    values: dict[str, str] = {}
    for line in env_path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or "=" not in stripped:
            continue
        key, value = stripped.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def env_value(key: str, file_values: dict[str, str], default: str = "") -> str:
    return os.getenv(key, file_values.get(key, default)).strip()


def redact(value: object, keep: int = 4) -> str:
    text = str(value or "")
    if not text:
        return "<missing>"
    if len(text) <= keep * 2:
        return "*" * len(text)
    return f"{text[:keep]}...{text[-keep:]}"


def redact_nested(value: object) -> object:
    sensitive = {"account_id", "accountId", "account_number", "accountNumber", "token"}
    if isinstance(value, dict):
        return {
            key: redact(val, keep=3) if key in sensitive else redact_nested(val)
            for key, val in value.items()
        }
    if isinstance(value, list):
        return [redact_nested(item) for item in value]
    return value


def save_json(name: str, payload: object) -> Path:
    path = OUTPUT_DIR / f"{name}.json"
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path


def save_records_csv(name: str, payload: object) -> pd.DataFrame:
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        records = payload.get("data", payload)
    else:
        records = []
    if isinstance(records, dict):
        records = [records]
    frame = pd.DataFrame(records)
    if not frame.empty:
        frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    return frame


env_file = Path(".env.webull-th") if Path(".env.webull-th").exists() else Path(".env")
file_values = read_env_file(env_file)

APP_KEY = env_value("WEBULL_APP_KEY", file_values)
APP_SECRET = env_value("WEBULL_APP_SECRET", file_values)
ACCESS_TOKEN = env_value("WEBULL_ACCESS_TOKEN", file_values)
ACCOUNT_ID = env_value("WEBULL_ACCOUNT_ID", file_values)

if LIVE_MODE and (not APP_KEY or not APP_SECRET):
    raise RuntimeError("Live mode requires WEBULL_APP_KEY and WEBULL_APP_SECRET")

print(
    {
        "live_mode": LIVE_MODE,
        "base_url": BASE_URL,
        "env_file": str(env_file),
        "app_key": redact(APP_KEY),
        "secret_present": bool(APP_SECRET),
        "token_present": bool(ACCESS_TOKEN),
        "account_id": redact(ACCOUNT_ID),
        "output_dir": str(OUTPUT_DIR),
    }
)


## Endpoint Map

หมวดนี้โฟกัส: Order Preview only; no Place, Replace, or Cancel execution


In [ ]:
pd.DataFrame(
    [
        {
            "name": endpoint["name"],
            "method": endpoint["method"],
            "path": endpoint["path"],
            "purpose": endpoint["purpose"],
        }
        for endpoint in ENDPOINTS
    ]
)


## Offline Samples

Cell ถัดไปคือ sample response สำหรับโหมด offline. โครงสร้างนี้ใช้เพื่อสอนการอ่าน JSON,
ไม่ใช่ข้อมูลตลาดแบบ real-time.


In [ ]:
SAMPLE_RESPONSES = {'order_preview': {'account_id': 'offline-account-001',
                   'symbol': 'AAPL',
                   'side': 'BUY',
                   'estimated_amount': '250.00',
                   'estimated_fee': '0.00',
                   'status': 'PREVIEW_ONLY'}}


## Request Helper

Helper นี้สร้าง signed request สำหรับ live mode ด้วย `HMAC-SHA256`.
ใน offline mode จะคืน sample response ทันที.


In [ ]:
def build_signature_headers(
    *,
    path: str,
    query_params: dict[str, object],
    body: dict[str, object] | None = None,
    algorithm: str = "HMAC-SHA256",
) -> dict[str, str]:
    timestamp = datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ")
    nonce = str(uuid.uuid4())
    body_str = json.dumps(body, separators=(",", ":"), ensure_ascii=False) if body else ""

    sign_params = {key: str(value) for key, value in query_params.items() if value is not None}
    sign_params.update(
        {
            "host": HOST,
            "x-app-key": APP_KEY,
            "x-signature-algorithm": algorithm,
            "x-signature-nonce": nonce,
            "x-signature-version": "1.0",
            "x-timestamp": timestamp,
        }
    )

    sorted_params = "&".join(f"{key}={value}" for key, value in sorted(sign_params.items()))
    string_to_sign = f"{path}&{sorted_params}"
    if body_str:
        body_md5 = hashlib.md5(body_str.encode()).hexdigest().upper()
        string_to_sign = f"{string_to_sign}&{body_md5}"

    encoded_string = quote(string_to_sign, safe="")
    digest = hashlib.sha256 if algorithm == "HMAC-SHA256" else hashlib.sha1
    signature = base64.b64encode(
        hmac.new((APP_SECRET + "&").encode(), encoded_string.encode(), digest).digest()
    ).decode()

    headers = {
        "Accept": "application/json",
        "Accept-Encoding": "gzip",
        "Content-Type": "application/json",
        "x-version": "v2",
        "x-app-key": APP_KEY,
        "x-timestamp": timestamp,
        "x-signature-version": "1.0",
        "x-signature-algorithm": algorithm,
        "x-signature-nonce": nonce,
        "x-signature": signature,
        "x-webull-client-source": "sdk",
    }
    if ACCESS_TOKEN:
        headers["x-access-token"] = ACCESS_TOKEN
    return headers


def call_endpoint(
    endpoint: dict[str, object],
    *,
    sample: object,
    query_params: dict[str, object] | None = None,
    body: dict[str, object] | None = None,
) -> object:
    if not LIVE_MODE:
        return sample

    query = {key: value for key, value in (query_params or {}).items() if value is not None}
    request_body = body or None
    path = str(endpoint["path"])
    headers = build_signature_headers(path=path, query_params=query, body=request_body)
    method = str(endpoint["method"]).upper()
    response = requests.request(
        method,
        f"{BASE_URL}{path}",
        headers=headers,
        params=query,
        json=request_body,
        timeout=30,
    )
    try:
        payload = response.json()
    except ValueError:
        payload = {"raw_text": response.text}
    if response.status_code >= 400:
        raise RuntimeError({"status_code": response.status_code, "payload": payload})
    return payload


print("request helpers ready")


## Run Endpoints

ทุก response จะถูก redacted แล้ว save เป็น JSON ใน `OUTPUT_DIR`.


In [ ]:
def resolve_endpoint_value(value: object) -> object:
    if value == "from_env_or_sample":
        return ACCOUNT_ID or "offline-account-001"
    if value == "from_env":
        return APP_KEY or "offline-app-key"
    if isinstance(value, dict):
        return {key: resolve_endpoint_value(item) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve_endpoint_value(item) for item in value]
    return value


results = {}
saved_paths = {}

for endpoint in ENDPOINTS:
    name = endpoint["name"]
    query_params = resolve_endpoint_value(endpoint.get("params") or {})
    body = resolve_endpoint_value(endpoint.get("body"))
    payload = call_endpoint(
        endpoint,
        sample=SAMPLE_RESPONSES[name],
        query_params=query_params,
        body=body,
    )
    clean_payload = redact_nested(payload)
    results[name] = clean_payload
    saved_paths[name] = str(save_json(name, clean_payload))

print({"endpoints": list(results), "saved_paths": saved_paths})


## Inspect and Export


In [ ]:
preview = pd.DataFrame([results["order_preview"]])
preview.to_csv(OUTPUT_DIR / "order-preview.csv", index=False)
preview


## Guardrails and Non-Goals

            - ไม่มี cell ใดใน notebook นี้เรียก Order Place, Order Replace, หรือ Order Cancel
- Preview ไม่ได้แปลว่า order จะส่งจริงหรือ fill จริง
- ถ้าจะเพิ่ม live order ในอนาคต ต้องแยก notebook และมี explicit confirmation gate


## Exercises

            1. เปลี่ยน quantity ใน body เป็น 2 แล้วดู estimated_amount ใน offline sample
2. เพิ่ม validation ว่า quantity ต้องมากกว่า 0 ก่อน preview
3. เขียน checklist ว่าต้องตรวจอะไรบ้างก่อนอนุญาต live order
